# Домашнее задание к занятию "Улучшение качества модели"

Выполнил: Ярослав Золотухин

## Импортируем данные

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

data = pd.read_csv( 'heart.csv' )
data.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [3]:
# категориальные переменные
data_one_hot_encoding = pd.get_dummies( data, columns = [ 'Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope' ] )

X = data_one_hot_encoding.drop(columns=['HeartDisease'])
y = data_one_hot_encoding['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Обучаем модель логичтической регрессии

In [5]:
from sklearn.model_selection import cross_validate

model_log = LogisticRegression(random_state=10).fit(X_train, y_train)
cross_validate(model_log, X_train, y_train, cv=10, scoring=['accuracy','recall','precision','f1'])

/Users/acer/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/acer/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#log

{'fit_time': array([0.02129889, 0.00958967, 0.0095973 , 0.01052117, 0.00922179,
        0.00883985, 0.00852704, 0.00861216, 0.00826812, 0.00812626]),
 'score_time': array([0.01418805, 0.0054121 , 0.00591278, 0.00550103, 0.00593114,
        0.00489807, 0.00498581, 0.00478697, 0.00503778, 0.00480795]),
 'test_accuracy': array([0.93243243, 0.87837838, 0.89189189, 0.86486486, 0.84931507,
        0.89041096, 0.78082192, 0.87671233, 0.89041096, 0.83561644]),
 'test_recall': array([0.925     , 0.875     , 0.875     , 0.90243902, 0.9       ,
        0.925     , 0.825     , 0.9       , 0.925     , 0.9       ]),
 'test_precision': array([0.94871795, 0.8974359 , 0.92105263, 0.86046512, 0.8372093 ,
        0.88095238, 0.78571429, 0.87804878, 0.88095238, 0.81818182]),
 'test_f1': array([0.93670886, 0.88607595, 0.8974359 , 0.88095238, 0.86746988,
        0.90243902, 0.80487805, 0.88888889, 0.90243902, 0.85714286])}

## GridSearchCV

Для пребора возьмем 3 параметра модели: C, l1_ratio и class_weight

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
        "model__solver": ["saga"],
        "model__penalty": ["elasticnet"],
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
        "model__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
        "model__class_weight": [None, "balanced"],
    }

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=10))
])

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=10, 
    scoring='accuracy'
)

grid.fit(X_train, y_train)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(max_iter=5000,
                                                           random_state=10))]),
             param_grid={'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'model__class_weight': [None, 'balanced'],
                         'model__l1_ratio': [0.0, 0.25, 0.5, 0.75, 1.0],
                         'model__penalty': ['elasticnet'],
                         'model__solver': ['saga']},
             scoring='accuracy')

In [10]:
cross_validate(grid.best_estimator_, X_train, y_train, cv=10, scoring=['accuracy','recall','precision','f1'])

{'fit_time': array([0.01020598, 0.00582385, 0.00491786, 0.00453973, 0.00430918,
        0.00567007, 0.00399113, 0.003721  , 0.00355816, 0.0038259 ]),
 'score_time': array([0.00683522, 0.00388932, 0.0038631 , 0.00328231, 0.00316501,
        0.00304198, 0.00294495, 0.00310493, 0.00281072, 0.00322008]),
 'test_accuracy': array([0.91891892, 0.86486486, 0.89189189, 0.86486486, 0.83561644,
        0.89041096, 0.79452055, 0.87671233, 0.90410959, 0.83561644]),
 'test_recall': array([0.925     , 0.85      , 0.875     , 0.90243902, 0.875     ,
        0.925     , 0.85      , 0.9       , 0.95      , 0.875     ]),
 'test_precision': array([0.925     , 0.89473684, 0.92105263, 0.86046512, 0.83333333,
        0.88095238, 0.79069767, 0.87804878, 0.88372093, 0.83333333]),
 'test_f1': array([0.925     , 0.87179487, 0.8974359 , 0.88095238, 0.85365854,
        0.90243902, 0.81927711, 0.88888889, 0.91566265, 0.85365854])}

## RandomizedSearchCV

In [11]:
from sklearn.model_selection import RandomizedSearchCV

randmize = RandomizedSearchCV(
    pipe,
    param_grid,
    cv=10, 
    scoring='accuracy'
)

randmize.fit(X_train, y_train)

RandomizedSearchCV(cv=10,
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('model',
                                              LogisticRegression(max_iter=5000,
                                                                 random_state=10))]),
                   param_distributions={'model__C': [0.001, 0.01, 0.1, 1, 10,
                                                     100],
                                        'model__class_weight': [None,
                                                                'balanced'],
                                        'model__l1_ratio': [0.0, 0.25, 0.5,
                                                            0.75, 1.0],
                                        'model__penalty': ['elasticnet'],
                                        'model__solver': ['saga']},
                   scoring='accuracy')

In [12]:
cross_validate(randmize.best_estimator_, X_train, y_train, cv=10, scoring=['accuracy','recall','precision','f1'])

{'fit_time': array([0.00929379, 0.00512886, 0.00482106, 0.00489092, 0.00452399,
        0.00581503, 0.00392008, 0.00344682, 0.00331306, 0.0030899 ]),
 'score_time': array([0.00705123, 0.00454497, 0.00442004, 0.00430226, 0.00520205,
        0.00525713, 0.00342107, 0.00319409, 0.00283003, 0.00281191]),
 'test_accuracy': array([0.91891892, 0.86486486, 0.90540541, 0.86486486, 0.8630137 ,
        0.87671233, 0.80821918, 0.89041096, 0.87671233, 0.80821918]),
 'test_recall': array([0.925     , 0.85      , 0.9       , 0.92682927, 0.9       ,
        0.9       , 0.9       , 0.925     , 0.95      , 0.85      ]),
 'test_precision': array([0.925     , 0.89473684, 0.92307692, 0.84444444, 0.85714286,
        0.87804878, 0.7826087 , 0.88095238, 0.84444444, 0.80952381]),
 'test_f1': array([0.925     , 0.87179487, 0.91139241, 0.88372093, 0.87804878,
        0.88888889, 0.8372093 , 0.90243902, 0.89411765, 0.82926829])}

## Выводы по проделанной работе

### a) Сравнение метрик построенных моделей

В работе были построены и сравнены несколько вариантов логистической регрессии:

| Модель | Accuracy | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|
| LogisticRegression | 0.869 | 0.871 | 0.895 | 0.882 |
| LogisticRegression + GridSearchCV | 0.868 | 0.870 | 0.893 | 0.881 |
| LogisticRegression + RandomizedSearchCV | 0.868 | 0.864 | 0.903 | 0.882 |

По результатам кросс-валидации все три модели показали очень близкое качество. Подбор гиперпараметров через `GridSearchCV` и `RandomizedSearchCV` не дал заметного прироста по `accuracy` и `f1-score` относительно базовой логистической регрессии.

При этом `RandomizedSearchCV` показал немного лучший `recall`, то есть такая модель лучше находит объекты положительного класса — пациентов с заболеванием сердца. Однако рост `recall` сопровождается небольшим снижением `precision`.

### b) Сравнение с результатами из ДЗ по теме «Ансамблирование»

В домашнем задании по ансамблированию были получены следующие результаты:

| Модель | Accuracy | F1-score класс 0 | F1-score класс 1 |
|---|---:|---:|---:|
| DecisionTreeClassifier | 0.78 | 0.76 | 0.80 |
| RandomForestClassifier | 0.88 | 0.85 | 0.89 |
| BaggingClassifier | 0.82 | 0.80 | 0.83 |
| StackingClassifier | 0.82 | 0.80 | 0.83 |

Лучший результат среди ансамблевых моделей показал `RandomForestClassifier`: accuracy = 0.88 и F1-score для класса 1 = 0.89. Это сопоставимо с результатами логистической регрессии после подбора гиперпараметров, где средний F1-score находится около 0.88.

Таким образом, улучшение качества логистической регрессии с помощью масштабирования признаков и подбора гиперпараметров позволило получить качество, близкое к лучшей ансамблевой модели. Однако `RandomForestClassifier` остается одним из наиболее сильных вариантов, так как он показал высокое качество без дополнительного подбора параметров.

Итог: для данной задачи хорошо работают как логистическая регрессия с предварительной обработкой признаков, так и ансамблевые методы. Наиболее стабильными и качественными моделями можно считать `RandomForestClassifier` и логистическую регрессию после настройки гиперпараметров.